# Phase 6 — Hybrid Recommender
Weighted blend of Item-Item CF and Sentence-Transformer content scores.
Alpha is tuned on the validation set.

In [ ]:
import sys, time
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.cf import ItemItemCF
from src.content import SentenceTransformerRecommender
from src.data import load_movies
from src.hybrid import HybridRecommender, _cf_scores_full, _content_scores_full, _normalize

In [ ]:
train  = pd.read_csv('../data/train.csv')
val    = pd.read_csv('../data/val.csv')
movies = load_movies()
print(f'train: {train.shape}  val: {val.shape}')

## 1. Fit Models

In [ ]:
t0   = time.time()
iicf = ItemItemCF(K=50).fit(train)
print(f'ItemItemCF fit in {time.time()-t0:.1f}s')

t0 = time.time()
st = SentenceTransformerRecommender().fit(movies, train)
print(f'SentenceTransformer fit in {time.time()-t0:.1f}s')

## 2. Sample Recommendations (alpha=0.6)

In [ ]:
USER_ID = 1
hybrid  = HybridRecommender(iicf, st, alpha=0.6)
recs    = hybrid.recommend(USER_ID, train, n=10)

rec_df = pd.DataFrame(recs, columns=['movieId', 'score'])
rec_df = rec_df.merge(movies[['movieId', 'title', 'genres']], on='movieId')
print(f'Hybrid recs for user {USER_ID} (alpha=0.6):')
rec_df

## 3. Alpha Tuning on Validation Set
Precompute CF + content score dicts once per user, then sweep alpha efficiently.

In [ ]:
K        = 10
N_USERS  = 200
alphas   = np.round(np.arange(0.0, 1.05, 0.1), 2)

val_items   = val.groupby('userId')['movieId'].apply(set).to_dict()
sample_users = list(val_items.keys())[:N_USERS]

p_at_k = {a: [] for a in alphas}

for uid in sample_users:
    cf_dict = _cf_scores_full(iicf, uid)
    ct_dict = _content_scores_full(st, uid, train)
    common  = list(set(cf_dict) & set(ct_dict))
    if not common:
        continue

    cf_norm = _normalize(np.array([cf_dict[m] for m in common]))
    ct_norm = _normalize(np.array([ct_dict[m] for m in common]))
    relevant = val_items.get(uid, set())

    for a in alphas:
        hybrid_scores = a * cf_norm + (1 - a) * ct_norm
        top_idx = np.argsort(hybrid_scores)[::-1][:K]
        rec_ids = {common[i] for i in top_idx}
        p_at_k[a].append(len(rec_ids & relevant) / K)

mean_p = {a: float(np.mean(v)) for a, v in p_at_k.items() if v}
best_alpha = max(mean_p, key=mean_p.get)
print(f'Best alpha: {best_alpha:.1f}  P@10: {mean_p[best_alpha]:.4f}')
print()
for a, p in sorted(mean_p.items()):
    print(f'  alpha={a:.1f}  P@10={p:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(list(mean_p.keys()), list(mean_p.values()), marker='o')
plt.axvline(best_alpha, color='red', linestyle='--', label=f'best alpha={best_alpha:.1f}')
plt.xlabel('alpha  (1.0 = pure CF,  0.0 = pure content)')
plt.ylabel('Precision@10')
plt.title('Hybrid alpha tuning on validation set')
plt.legend()
plt.tight_layout()
plt.savefig('../data/hybrid_alpha_tuning.png', dpi=100)
plt.show()
print('Saved to data/hybrid_alpha_tuning.png')

## 4. Best-Alpha Evaluation

In [ ]:
best_hybrid = HybridRecommender(iicf, st, alpha=best_alpha)

hits = []
for uid in sample_users:
    recs    = best_hybrid.recommend(uid, train, n=K)
    rec_ids = {r[0] for r in recs}
    hits.append(len(rec_ids & val_items.get(uid, set())) / K)

p_hybrid = float(np.mean(hits))
print(f'Hybrid (alpha={best_alpha:.1f})  P@10: {p_hybrid:.4f}')

## 5. Summary

| Model | P@10 |
|---|---|
| Item-Item CF | 0.0855 |
| TF-IDF Content | 0.0160 |
| Matrix Factorization | 0.0070 |
| Sentence-Transformer | 0.0060 |
| User-User CF | 0.0015 |
| **Hybrid (best alpha)** | _run to see_ |

The alpha sweep shows how CF dominates when alpha→1 (item-item CF has the strongest signal on MovieLens-1M).
Content signal provides a small lift for users with sparse CF history (cold-start boundary).